# INX Future Inc — Employee Performance Analysis
## Notebook 1: Data Processing
**Project Code:** 10281 | **IABAC Certified Data Scientist Project**

This notebook covers loading, cleaning, encoding, and saving the processed dataset ready for modeling.

---
## 1. Import Libraries

In [12]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully')

Libraries imported successfully


---
## 2. Load Raw Data

In [13]:
df = pd.read_excel('../../data/raw/INX_Future_Inc_Employee_Performance_CDS_Project2_Data_V1_8.xls')
print(f'Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

Dataset loaded: 1200 rows, 28 columns


,EmpNumber,Age,Gender,EducationBackground,MaritalStatus,EmpDepartment,EmpJobRole,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,...,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition,PerformanceRating
0,E1001000,32,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,10,3,...,4,10,2,2,10,7,0,8,No,3
1,E1001006,47,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,14,4,...,4,20,2,3,7,7,1,7,No,3
2,E1001007,40,Male,Life Sciences,Married,Sales,Sales Executive,Travel_Frequently,5,4,...,3,20,2,3,18,13,1,12,No,4
3,E1001009,41,Male,Human Resources,Divorced,Human Resources,Manager,Travel_Rarely,10,4,...,2,23,2,2,21,6,12,6,No,3
4,E1001010,60,Male,Marketing,Single,Sales,Sales Executive,Travel_Rarely,16,4,...,4,10,1,3,2,2,2,2,No,3


---
## 3. Initial Data Inspection

In [14]:
print('=== Dataset Info ===')
df.info()

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   EmpNumber                     1200 non-null   object
 1   Age                           1200 non-null   int64 
 2   Gender                        1200 non-null   object
 3   EducationBackground           1200 non-null   object
 4   MaritalStatus                 1200 non-null   object
 5   EmpDepartment                 1200 non-null   object
 6   EmpJobRole                    1200 non-null   object
 7   BusinessTravelFrequency       1200 non-null   object
 8   DistanceFromHome              1200 non-null   int64 
 9   EmpEducationLevel             1200 non-null   int64 
 10  EmpEnvironmentSatisfaction    1200 non-null   int64 
 11  EmpHourlyRate                 1200 non-null   int64 
 12  EmpJobInvolvement             1200 non-null   int64 
 1

In [15]:
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found — clean dataset.')

=== Missing Values ===
No missing values found — clean dataset.


In [16]:
print('=== Duplicate Rows ===')
dupes = df.duplicated().sum()
print(f'{dupes} duplicate rows found.')
if dupes > 0:
    df = df.drop_duplicates()
    print(f'Dropped duplicates. New shape: {df.shape}')

=== Duplicate Rows ===
0 duplicate rows found.


In [17]:
print('=== Target Variable Distribution ===')
perf_counts = df['PerformanceRating'].value_counts().sort_index()
perf_pct = df['PerformanceRating'].value_counts(normalize=True).sort_index() * 100
perf_summary = pd.DataFrame({'Count': perf_counts, 'Percentage (%)': perf_pct.round(2)})
rating_map = {2: 'Low', 3: 'Good', 4: 'Excellent'}
perf_summary.index = [f"{i} ({rating_map[i]})" for i in perf_summary.index]
print(perf_summary)

=== Target Variable Distribution ===
               Count  Percentage (%)
2 (Low)          194           16.17
3 (Good)         874           72.83
4 (Excellent)    132           11.00


---
## 4. Drop Non-Predictive Columns

`EmpNumber` is a unique identifier — it carries no predictive signal and must be dropped before modeling.

In [18]:
df = df.drop(columns=['EmpNumber'])
print(f'EmpNumber dropped. Shape: {df.shape}')

EmpNumber dropped. Shape: (1200, 27)


---
## 5. Identify Column Types

In [19]:
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols = [c for c in numerical_cols if c != 'PerformanceRating']

print('Categorical columns:', categorical_cols)
print('\nNumerical columns:', numerical_cols)

Categorical columns: ['Gender', 'EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole', 'BusinessTravelFrequency', 'OverTime', 'Attrition']

Numerical columns: ['Age', 'DistanceFromHome', 'EmpEducationLevel', 'EmpEnvironmentSatisfaction', 'EmpHourlyRate', 'EmpJobInvolvement', 'EmpJobLevel', 'EmpJobSatisfaction', 'NumCompaniesWorked', 'EmpLastSalaryHikePercent', 'EmpRelationshipSatisfaction', 'TotalWorkExperienceInYears', 'TrainingTimesLastYear', 'EmpWorkLifeBalance', 'ExperienceYearsAtThisCompany', 'ExperienceYearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


---
## 6. Encode Categorical Variables

We use **Label Encoding** for binary/ordinal columns and **One-Hot Encoding** for nominal columns with more than 2 categories.

In [20]:
from sklearn.preprocessing import LabelEncoder

# Binary columns — Label Encode directly
binary_cols = ['Gender', 'OverTime', 'Attrition']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])
    print(f'{col} encoded.')

# Ordinal column — BusinessTravelFrequency
travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
df['BusinessTravelFrequency'] = df['BusinessTravelFrequency'].map(travel_map)
print('BusinessTravelFrequency ordinal encoded.')

Gender encoded.
OverTime encoded.
Attrition encoded.
BusinessTravelFrequency ordinal encoded.


In [21]:
# Nominal columns — One-Hot Encode
nominal_cols = ['EducationBackground', 'MaritalStatus', 'EmpDepartment', 'EmpJobRole']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=False)
print(f'One-Hot Encoding applied. New shape: {df.shape}')
df.head(2)

One-Hot Encoding applied. New shape: (1200, 57)


,Age,Gender,BusinessTravelFrequency,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,...,EmpJobRole_Manager R&D,EmpJobRole_Manufacturing Director,EmpJobRole_Research Director,EmpJobRole_Research Scientist,EmpJobRole_Sales Executive,EmpJobRole_Sales Representative,EmpJobRole_Senior Developer,EmpJobRole_Senior Manager R&D,EmpJobRole_Technical Architect,EmpJobRole_Technical Lead
0,32,1,1,10,3,4,55,3,2,4,...,False,False,False,False,True,False,False,False,False,False
1,47,1,1,14,4,4,42,3,2,1,...,False,False,False,False,True,False,False,False,False,False


---
## 7. Feature Scaling

Standardization (zero mean, unit variance) is applied to all numerical features. This is critical for distance-based and gradient-based algorithms.

In [22]:
from sklearn.preprocessing import StandardScaler
import pickle

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Save scaler for use during prediction
with open('../../data/processed/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('StandardScaler applied and saved.')
df[numerical_cols].describe().round(2)

StandardScaler applied and saved.


,Age,DistanceFromHome,EmpEducationLevel,EmpEnvironmentSatisfaction,EmpHourlyRate,EmpJobInvolvement,EmpJobLevel,EmpJobSatisfaction,NumCompaniesWorked,EmpLastSalaryHikePercent,EmpRelationshipSatisfaction,TotalWorkExperienceInYears,TrainingTimesLastYear,EmpWorkLifeBalance,ExperienceYearsAtThisCompany,ExperienceYearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
count,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00,1200.00
mean,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-2.08,-1.00,-1.81,-1.57,-1.78,-2.45,-0.96,-1.57,-1.08,-1.17,-1.60,-1.45,-2.21,-2.49,-1.14,-1.19,-0.68,-1.16
25%,-0.76,-0.88,-0.86,-0.66,-0.89,-1.04,-0.96,-0.67,-0.67,-0.89,-0.67,-0.68,-0.62,-1.06,-0.65,-0.63,-0.68,-0.59
50%,-0.10,-0.26,0.10,0.26,0.00,0.38,-0.06,0.24,-0.27,-0.34,0.26,-0.17,0.17,0.37,-0.33,-0.36,-0.37,-0.31
75%,0.67,0.59,1.06,1.18,0.84,0.38,0.84,1.15,0.54,0.77,1.19,0.47,0.17,0.37,0.47,0.75,0.25,0.82
max,2.54,2.43,2.02,1.18,1.68,1.79,2.65,1.15,2.57,2.70,1.19,3.68,2.55,1.80,5.28,3.79,3.98,3.64


---
## 8. Save Processed Dataset

In [23]:
df.to_csv('../../data/processed/INX_Employee_Processed.csv', index=False)
print(f'Processed dataset saved. Final shape: {df.shape}')
print(f'Total features: {df.shape[1] - 1} | Target: PerformanceRating')

Processed dataset saved. Final shape: (1200, 57)
Total features: 56 | Target: PerformanceRating


---
## 9. Summary

| Step | Action | Result |
|------|--------|--------|
| Load | Read XLS file | 1200 rows × 28 columns |
| Missing Values | Checked | None found |
| Duplicates | Checked & removed | Clean dataset |
| Drop | EmpNumber (ID column) | 27 columns remain |
| Encode | Binary + Ordinal + One-Hot | All categoricals converted |
| Scale | StandardScaler on numerical | Mean=0, Std=1 |
| Save | CSV to data/processed/ | Ready for modeling |